In [ ]:
from contextlib import contextmanager
from pathlib import Path
from typing import Iterator

import pandas as pd
import shutil
import tempfile
from urllib.request import urlopen, Request
import zipfile

In [ ]:
BASE_URL = "https://portaldatransparencia.gov.br/download-de-dados/notas-fiscais"
DATASET_NAME = "notas-fiscais"
DEFAULT_CSV_KWARGS = {
    "sep": ";",
    "encoding": "latin1",
    "low_memory": False,
}
# Forçar colunas a serem String
DEFAULT_TEXT_COLUMNS = [
    "CHAVE DE ACESSO",
]

# Funções

## Baixar arquivo Zip da fonte de Notas Fiscais

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    """Return the nearest parent folder containing pyproject.toml."""
    current = (start or Path.cwd()).resolve()

    for path in (current, *current.parents):
        if (path / "pyproject.toml").exists():
            return path

    raise FileNotFoundError("Nao encontrei pyproject.toml nos diretorios pais.")


def formatar_competencia(ano: int, mes: int) -> tuple[str, str]:
    """Return competencia values as yyyyMM and yyyy-MM."""
    ano = int(ano)
    mes = int(mes)

    if not 1 <= mes <= 12:
        raise ValueError("mes deve estar entre 1 e 12.")

    return f"{ano:04d}{mes:02d}", f"{ano:04d}-{mes:02d}"


def dataset_dir(stage: str, dataset_name: str = DATASET_NAME) -> Path:
    """Return data/<stage>/<dataset_name>, creating it when needed."""
    path = find_project_root() / "data" / stage / dataset_name
    path.mkdir(parents=True, exist_ok=True)
    return path


def download_file(url: str, destination: Path, chunk_size: int = 1024 * 1024) -> None:
    """Download a URL to destination, using requests when available."""
    try:
        import requests
    except ImportError:
        request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(request) as response, destination.open("wb") as file:
            while chunk := response.read(chunk_size):
                file.write(chunk)
        return

    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with destination.open("wb") as file:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    file.write(chunk)


def safe_extract_zip(zip_path: Path, destination: Path) -> None:
    """Extract zip_path without allowing files to escape destination."""
    destination = destination.resolve()

    with zipfile.ZipFile(zip_path) as zip_file:
        for member in zip_file.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Arquivo inseguro dentro do zip: {member.filename}")

        zip_file.extractall(destination)


def baixar_zip_notas_fiscais(
    ano: int,
    mes: int,
    raw_dir: Path | str | None = None,
    overwrite: bool = True,
) -> Path:
    """Baixa o zip bruto em data/raw/notas-fiscais/yyyy-MM.zip."""
    competencia, competencia_com_hifen = formatar_competencia(ano, mes)
    url = f"{BASE_URL}/{competencia}"
    raw_path = Path(raw_dir) if raw_dir is not None else dataset_dir("raw")
    raw_path.mkdir(parents=True, exist_ok=True)
    destino_final = raw_path / f"{competencia_com_hifen}.zip"

    if destino_final.exists() and not overwrite:
        print(f"Zip ja existe: {destino_final}")
        return destino_final

    with tempfile.TemporaryDirectory() as tmp:
        zip_path = Path(tmp) / f"{competencia}.zip"
        print(f"Baixando {url}...")
        download_file(url, zip_path)

        if destino_final.exists():
            destino_final.unlink()
        shutil.move(str(zip_path), str(destino_final))

    print(f"Zip salvo em: {destino_final}")
    return destino_final


def listar_arquivos_zip(zip_path: Path | str) -> list[str]:
    """Lista os arquivos contidos em um zip."""
    with zipfile.ZipFile(zip_path) as zip_file:
        return zip_file.namelist()

## Extrair notas fiscais e converter processar em parquets

In [ ]:
@contextmanager
def extrair_zip_temporario(zip_path: Path | str) -> Iterator[Path]:
    """Extrai um zip em pasta temporaria e limpa tudo ao sair do bloco."""
    with tempfile.TemporaryDirectory() as tmp:
        extract_path = Path(tmp) / "extracted"
        extract_path.mkdir()
        safe_extract_zip(Path(zip_path), extract_path)
        yield extract_path


def encontrar_csvs(pasta: Path | str) -> list[Path]:
    """Encontra todos os CSVs extraidos."""
    csvs = sorted(Path(pasta).rglob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"Nao encontrei arquivos CSV em {pasta}.")
    return csvs


def ler_csvs_notas_fiscais(
    pasta_extraida: Path | str,
    csv_kwargs: dict | None = None,
    colunas_texto: list[str] | None = None,
) -> pd.DataFrame:
    """Le todos os CSVs extraidos e concatena em um unico DataFrame."""
    kwargs = DEFAULT_CSV_KWARGS.copy()
    if csv_kwargs:
        kwargs.update(csv_kwargs)

    dtype = dict(kwargs.pop("dtype", {}) or {})
    for coluna in (colunas_texto or DEFAULT_TEXT_COLUMNS):
        dtype.setdefault(coluna, "string")
    kwargs["dtype"] = dtype

    dataframes = []
    for csv_path in encontrar_csvs(pasta_extraida):
        df = pd.read_csv(csv_path, **kwargs)
        df["arquivo_origem"] = csv_path.name
        dataframes.append(df)

    return pd.concat(dataframes, ignore_index=True)


def preprocessar_notas_fiscais(
    df: pd.DataFrame,
    colunas: list[str],
    colunas_texto: list[str] | None = None,
) -> pd.DataFrame:
    """Seleciona colunas e agrega linhas duplicadas em uma coluna frequencia."""
    colunas_faltantes = [coluna for coluna in colunas if coluna not in df.columns]
    if colunas_faltantes:
        raise KeyError(f"Colunas ausentes no DataFrame: {colunas_faltantes}")

    df_selecionado = df[colunas].copy()
    for coluna in (colunas_texto or DEFAULT_TEXT_COLUMNS):
        if coluna in df_selecionado.columns:
            df_selecionado[coluna] = df_selecionado[coluna].astype("string")

    return (
        df_selecionado
        .value_counts(dropna=False)
        .reset_index(name="frequencia")
    )

    def processar_mes_notas_fiscais(
    ano: int,
    mes: int,
    colunas: list[str],
    overwrite_raw: bool = True,
    overwrite_processed: bool = True,
    csv_kwargs: dict | None = None,
    colunas_texto: list[str] | None = None,
) -> Path:
    """Pipeline completo: raw zip -> extracao temporaria -> parquet processado."""
    _, competencia_com_hifen = formatar_competencia(ano, mes)
    processed_path = dataset_dir("processed") / f"{competencia_com_hifen}.parquet"

    if processed_path.exists() and not overwrite_processed:
        print(f"Parquet ja existe: {processed_path}")
        return processed_path

    zip_path = baixar_zip_notas_fiscais(
        ano,
        mes,
        overwrite=overwrite_raw,
    )

    print(f"Extraindo temporariamente {zip_path.name}...")
    with extrair_zip_temporario(zip_path) as pasta_extraida:
        df = ler_csvs_notas_fiscais(
            pasta_extraida,
            csv_kwargs=csv_kwargs,
            colunas_texto=colunas_texto,
        )
        df_processado = preprocessar_notas_fiscais(
            df,
            colunas=colunas,
            colunas_texto=colunas_texto,
        )

    if processed_path.exists():
        processed_path.unlink()
    df_processado.to_parquet(processed_path, index=False)

    print(f"Parquet salvo em: {processed_path}")
    return processed_path


# Main

Colunas disponíveis:
```
Index(['CHAVE DE ACESSO', 'MODELO', 'SÉRIE', 'NÚMERO', 'NATUREZA DA OPERAÇÃO',
       'DATA EMISSÃO', 'CPF/CNPJ Emitente', 'RAZÃO SOCIAL EMITENTE',
       'INSCRIÇÃO ESTADUAL EMITENTE', 'UF EMITENTE', 'MUNICÍPIO EMITENTE',
       'CÓDIGO ÓRGÃO SUPERIOR DESTINATÁRIO', 'ÓRGÃO SUPERIOR DESTINATÁRIO',
       'CÓDIGO ÓRGÃO DESTINATÁRIO', 'ÓRGÃO DESTINATÁRIO', 'CNPJ DESTINATÁRIO',
       'NOME DESTINATÁRIO', 'UF DESTINATÁRIO', 'INDICADOR IE DESTINATÁRIO',
       'DESTINO DA OPERAÇÃO', 'CONSUMIDOR FINAL', 'PRESENÇA DO COMPRADOR',
       'NÚMERO PRODUTO', 'DESCRIÇÃO DO PRODUTO/SERVIÇO', 'CÓDIGO NCM/SH',
       'NCM/SH (TIPO DE PRODUTO)', 'CFOP', 'QUANTIDADE', 'UNIDADE',
       'VALOR UNITÁRIO', 'VALOR TOTAL'],
      dtype='str')
```

In [ ]:
ano = 2025
mes = 01

In [ ]:
# primeiro baixa apenas o ZIP bruto.
zip_path = baixar_zip_notas_fiscais(ano, mes)
listar_arquivos_zip(zip_path)[:10]

In [ ]:
# Depois escolha as colunas de interesse e rode o preprocessamento.
COLUNAS_INTERESSE = [
    "CHAVE DE ACESSO",
    "NÚMERO PRODUTO",
    "NATUREZA DA OPERAÇÃO",    
    "CONSUMIDOR FINAL",
    "PRESENÇA DO COMPRADOR",
    "DESCRIÇÃO DO PRODUTO/SERVIÇO",
    "CÓDIGO NCM/SH",
    "NCM/SH (TIPO DE PRODUTO)",
    "QUANTIDADE",
    "UNIDADE",
    "VALOR UNITÁRIO",
    "VALOR TOTAL",
]
processar_mes_notas_fiscais(ano, mes, COLUNAS_INTERESSE)


Baixando https://portaldatransparencia.gov.br/download-de-dados/notas-fiscais/202602...
Zip salvo em: C:\Users\samue\Scripts\Nofis-Classifier\data\raw\notas-fiscais\2026-02.zip
Baixando https://portaldatransparencia.gov.br/download-de-dados/notas-fiscais/202602...
Zip salvo em: C:\Users\samue\Scripts\Nofis-Classifier\data\raw\notas-fiscais\2026-02.zip
Extraindo temporariamente 2026-02.zip...
Parquet salvo em: C:\Users\samue\Scripts\Nofis-Classifier\data\processed\notas-fiscais\2026-02.parquet


WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/processed/notas-fiscais/2026-02.parquet')